# IoT-ASP Colab ETL

GCS → features → Gemini Enterprise / ADK. **No site PII.**

- Docs: [`docs/colab-gemini-pipeline.md`](../docs/colab-gemini-pipeline.md)
- Contract: [`docs/api-contract.md`](../docs/api-contract.md) `schemaVersion: 1`
- Shared SciPy anomaly: `services/autoroute-adk/iot_asp_autoroute/vib_anomaly.py` (1 Hz, quantum **0.0005 g**)
- Shared features: `iot_asp_autoroute.colab_etl`

**Secrets:** Colab `userdata.get('GCP_SA_JSON')` only. Never paste SA JSON into git/chat or download to Studio/Laptop.

## 1. Offline bootstrap (runs on laptop / CI)

Adds the ADK package to `sys.path` so Colab and ADK share one SciPy path — no duplicate detector.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

# Repo root detection: notebook in notebooks/, or Colab clone layout
CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/IoT-ASP"),
    Path("/content"),
]
ROOT = None
for base in CANDIDATES:
    adk = base / "services" / "autoroute-adk"
    if (adk / "iot_asp_autoroute" / "vib_anomaly.py").is_file():
        ROOT = base
        if str(adk) not in sys.path:
            sys.path.insert(0, str(adk))
        break

assert ROOT is not None, (
    "Clone team-project-pikachu/IoT-ASP (or open from repo root) so "
    "services/autoroute-adk/iot_asp_autoroute is importable"
)

from iot_asp_autoroute.colab_etl import (
    TELEMETRY_FEATURE_COLUMNS,
    extract_features,
    offline_dry_run,
    sample_telemetry_fixture,
    sample_vib_gyro_sound_fixture,
    suggest_patch_stub,
)
from iot_asp_autoroute.vib_anomaly import VIB_QUANTUM, synthetic_demo_series

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "bear-iot-asp-rec")
ENGINE_ID = os.environ.get("IOT_ASP_GEMINI_ENGINE_ID", "iot-asp-autoroute")
LIVE_GCS = os.environ.get("LIVE_GCS", "0") == "1"

print("root", ROOT)
print("project", PROJECT, "engine", ENGINE_ID, "LIVE_GCS", LIVE_GCS)
print("VIB_QUANTUM", VIB_QUANTUM, "feature_cols", len(TELEMETRY_FEATURE_COLUMNS))

## 2. Feature extract + SciPy anomaly (shared with ADK)

Telemetry columns match the api-contract heartbeat. Anomaly uses `medfilt` + MAD + `find_peaks` from the shared module.

In [ ]:
tel = sample_telemetry_fixture("node1")
series = synthetic_demo_series(60)  # 1 Hz synthetic vib (g)
features = extract_features(tel, vib_series=series, engine_id=ENGINE_ID)

print("telemetry keys", sorted(features["telemetry"].keys()))
print("priorHint", features["derived"]["priorHint"])
print(
    "anomaly",
    "n=", features["anomaly"]["n"],
    "disturbance=", features["anomaly"]["disturbance"],
    "funcs=", features["anomaly"]["functions"],
)
print(json.dumps({"kind": features["kind"], "deviceId": features["deviceId"], "derived": features["derived"]}, indent=2))

## 2b. Synthetic vib + gyro + sound (offline)

Fixture covers accel axes, gyro ω, `micDiff = micEnergy − α·outLevel`, band energy &lt;20 Hz / &gt;17 kHz, and `soundBurst` / `extremeActive`. Patch stub should bias shriek/extreme.

In [ ]:
sensor_tel = sample_vib_gyro_sound_fixture("node1")
sensor_feat = extract_features(sensor_tel, vib_series=series, engine_id=ENGINE_ID)
sensor_sug = suggest_patch_stub(sensor_tel, engine_id=ENGINE_ID)

print("sensors", json.dumps(sensor_feat["derived"]["sensors"], indent=2))
print("micDiff", sensor_feat["telemetry"].get("micDiff"), "bandBurst", sensor_feat["telemetry"].get("bandBurst"))
print("shriekBias", sensor_sug.get("shriekBiasEligible"), "algo", (sensor_sug.get("suggestion") or {}).get("algo"))
assert sensor_sug.get("ok") and sensor_sug.get("shriekBiasEligible"), sensor_sug
assert (sensor_sug.get("suggestion") or {}).get("algo") in (
    "shriek_chirp", "shriek_sweep", "burst", "infra_mod",
    "cry_mirror", "siren_mirror", "death_metal_mirror",
), sensor_sug
print("synthetic vib+gyro+sound OK")


## 3. Patch **suggestion** stub (not authoritative)

ADK worker still clamps and writes `meta/patches/`. Hold/Manual must refuse.

In [ ]:
sug = suggest_patch_stub(tel, engine_id=ENGINE_ID)
hold = suggest_patch_stub({**tel, "holdManual": True}, engine_id=ENGINE_ID)
assert sug.get("ok") and sug.get("suggestion"), sug
assert hold.get("refused") is True, hold
print("suggestion algo", sug["suggestion"]["algo"])
print("holdManual refused", hold["refused"], hold.get("reason"))

dry = offline_dry_run()
assert dry["ok"], dry
print("offline_dry_run ok", dry["ok"], "missing", dry["missingTelemetryColumns"])

## 4. Live GCS (Colab only — gated)

Set Colab userdata: `GCP_SA_JSON`, `IOT_ASP_GCS_BUCKET`. Set env `LIVE_GCS=1` to enable. Reads `meta/telemetry/`, writes `meta/features/` only.

In [ ]:
def colab_auth_and_client():
    """Credential refs via userdata — never log SA JSON."""
    from google.colab import userdata  # type: ignore
    from google.oauth2 import service_account
    from google.cloud import storage

    bucket_name = userdata.get("IOT_ASP_GCS_BUCKET") or os.environ.get("IOT_ASP_GCS_BUCKET")
    sa_json = userdata.get("GCP_SA_JSON")
    if not bucket_name or not sa_json:
        raise RuntimeError("Set userdata IOT_ASP_GCS_BUCKET + GCP_SA_JSON (names only in docs)")
    creds = service_account.Credentials.from_service_account_info(json.loads(sa_json))
    client = storage.Client(project=PROJECT, credentials=creds)
    return client, bucket_name


def latest_telemetry(client, bucket_name: str, node_id: str):
    blobs = list(client.list_blobs(bucket_name, prefix=f"meta/telemetry/{node_id}/"))
    blobs = sorted(blobs, key=lambda b: b.name, reverse=True)
    if not blobs:
        return None
    return json.loads(blobs[0].download_as_text())


def write_features(client, bucket_name: str, features: dict) -> str:
    node = features["deviceId"]
    ts = str(features.get("ts") or "latest").replace(":", "").replace("/", "-")
    path = f"meta/features/{node}/{ts}.json"
    client.bucket(bucket_name).blob(path).upload_from_string(
        json.dumps(features, indent=2),
        content_type="application/json",
    )
    return f"gs://{bucket_name}/{path}"


if LIVE_GCS:
    client, bucket = colab_auth_and_client()
    node = "node1"
    remote = latest_telemetry(client, bucket, node) or tel
    feat = extract_features(remote, engine_id=ENGINE_ID)
    uri = write_features(client, bucket, feat)
    print("wrote", uri)
    # Optional: print suggestion only — do NOT write meta/patches from Colab
    print("suggestion", suggest_patch_stub(remote, engine_id=ENGINE_ID).get("suggestion", {}).get("algo"))
else:
    print("LIVE_GCS=0 — skip GCS (offline OK). On Colab: userdata + LIVE_GCS=1")